# __MEMORIA PRÁCTICA FINAL - INTELIGENCIA ARTIFICIAL EN LAS ORGANIZACIONES__
### *Análisis de variables y resultados en predicción de posiciones finales en Fórmula 1*
#### Grupo 9: Grado en Ingeniería Informática UC3M 2025/26
- Javier Rosales Lozano
- Manuel Roldán Matea
- Nahuel Sebastián Vargas
- Alejandro Rodríguez García

En el siguiente documento se presenta el desarrollo completo del proyecto de la práctica final de la asignatura Ingeligencia Artificial en las Organizaciones. La memoria recoge desde la definición del problema propuesto hasta el análisis de resultados y métricas/variables relevantes para la predicción.

A modo de guía del proyecto, la memoria se recoge en un archivo Jupyter Notebook (.ipynb) donde se encontrará documentado todo el proceso, junto con los fragmentos de código y ejecuciones que dan pie a los resultados obtenidos.

# 1. DEFINICIÓN DEL PROBLEMA

### 1.1. ESTADO DEL ARTE

La aplicación de Inteligencia Artificial (IA) en la Fórmula 1 (F1) ha evolucionado desde sistemas básicos de telemetría hacia modelos predictivos sofisticados que optimizan decisiones críticas en tiempo real durante la carrera. La F1 genera aproximadamente ~1.5TB de datos por carrera, incluyendo telemetría, cámaras 360º, sensores de neumáticos, e información mucho más específica, convirtiéndola en el laboratorio de IA más avanzado del motosport. Desde 2017 (reglamento híbrido), la IA pasó de análisis post-carrera a toma de decisiones en vivo; los equipos líderes utilizan tecnologías actuales para ejecutar miles de millones de simulaciones por fin de semana, analizando escenarios posibles en la carrera.

Una de estas aplicaciones se puede apreciar en estrategias actuales de éstos equipos líderes, como Red Bull Racing. Esta escudería utiliza la tecnología Oracle Cloud Infrastructure para intentar preveer los resultados de las carreras según la toma de decisiones. En 2021, uno de sus logros más relevantes fue ganar el Gran Premio de Abu Dhabi por estrategia de IA ([AI and Machine Learning: The future of Formula 1 - NextTech Today](https://nexttechtoday.com/featured/part-3-can-ai-outrace-instinct-in-f1/))

Otra escudería líder, Scuderia Ferrari, en colaboració con AWS, implementó su "ML Pit-Stop Optimizer", que revolucionó los tiempos de boxes al reducirlos de 2.4s (récord anterior) a un impresionante tiempo de 2.1s durante el Gran Premio de Miami en 2025. Este sistema de machine learning procesa en tiempo real datos de desgaste de neumáticos y tráfico en pista, optimizando la secuencia de mecánicos y reduciendo errores humanos en fracciones críticas de segundo ([AWS Machine Learning Supports Scuderia Ferrari HP Pitstop Analysis](https://aws.amazon.com/es/blogs/machine-learning/aws-machine-learning-supports-scuderia-ferrari-hp-pit-stop-analysis/))

En resumen, en la F1, la IA ha pasado de ser una herramienta analítica a un elemento decisivo en la competición, donde cada décima de segundo puede significar entre 3-5 posiciones en la clasificación final. Esta evolución establece el *ensemble learning* como estándar académico industrial, como demuestra [Rao (2023](https://norma.ncirl.ie/7601/) en su tésis doctoral *Predictive Model for Pitstop Strategy in Formula 1 using Ensemble Learninhg (National College of Ireland)*, que integra telemetría, clima y desgaste de neumáticos para predecir vueltas óptimas de pitstop con un 25% de error cuadrático medio mejor que baselines tradicionales.

Sin embargo, nuestro proyecto no se enfoca en un predictor de posiciones potenciales, sino en un modelo para analizar por qué se obtienen estos resultados en la carrera, y justificar las predicciones del mejor modelo con explicabilidad total. En otras palabras, el modelo no predice que Verstappen vaya a quedar primero, sino que epxlica por qué Verstappen acaba primero, teniendo en cuenta los factores que afectan a la carrera (como el circuito o clima), la estrategia que se sigue (agresividad/moderación, neumáticos) y métricas de tiempo (por ejemplo, los tiempos en boxes).

### 1.2. COMPLEJIDAD DEL PROBLEMA

Sin embargo, predecir posiciones finales en F1 es difícil. Imaginemos que somos estrategas de Red Bull en plena carrera de Mónaco, y tenemos un minuto para decidir la siguiente estrategia: ¿Pit-stop ahora o dentro de tres vueltas? ¿Neumáticos medios o duros? ¿Verstappen agresivo o conservador? Un error de medio segundo en boxes puede costar 4 posiciones.

- __Análisis Temporal__: Una carrera de F1 evoluciona dinámicamente durante ~2 horas; las regulaciones cambian anualmente y los equipos mejoran iterativamente. Si predecimos usando datos futuros (usar datos de 2024 para predecir 2020), el modelo aparenta genialidad en validación pero falla estrepitosamente en carrera real, además de que tendemos al Data Leakage. La solución para ello es realizar una partición de los datos histórica, entrenando con el pasado y prediciendo el futuro (2018-19 predicen 2020, 2018-20 predicen 2021...).

- __Representación de Datos__: En este contexto existen múltiples datos numéricos (temperatura de la pista/aire, viento...) combinados con diferentes actores en la carrera (22 pilotos, 10 equipos, 24 circuitos...). Esto genera un dataset masivo e inherentemente caótico, en el que la IA debe encontrar patrones. La solución se basará en un preprocesador automático construido a partir del mejor Escalador/Imputador de los datos, siguiendo la optimización con KNN y el análisis de series temporales definido anteriormente.

- __Análisis de Atributos__: La correlación de las variables a analizar puede confundir al modelo. El análisis exhaustivo de las variables y su aportación a la predicción de la variable objetivo es clave, de la misma manera que el tratamiento de valores faltantes o ruido en los datos. La solución aportada se basará en un análisis de multicolinealidad, donde detectaremos qué variables son más relevantes para el modelo.

- __Justificación y entendimiento de resultados__: Para comparar estrategias, estas escuderías necesitan confianza absoluta en la toma de decisiones. Estos modelos deben ser capaces de justificar y explicar a los equipos por qué se producen los resultados que otorga. Es por ello que realizaremos un análisis completo y consistente de los resultados para entender en qué se ha basado el modelo para obtener los resultados que ha dado.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
import shap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import randint, uniform
import warnings
import os

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
os.makedirs("plots", exist_ok=True)

In [ ]:
SEED = 6967
np.random.seed(SEED)

In [ ]:
print("\n=== PIPELINE COMPLETO F1 POSITION PREDICTOR ===\n")

In [ ]:
# 1. CARGA Y LIMPIEZA
print("=== 1. CARGA Y LIMPIEZA ===")
try:
    datos = pd.read_csv("data/f1_pitstops_2018_2024.csv", sep=",")
except:
    print("❌ Archivo no encontrado. Crea 'data/f1_pitstops_2018_2024.csv'")
    exit()
print(f"Tamaño inicial: {datos.shape}")

In [ ]:
# Filtrar DNFs y columnas innecesarias
vueltas_max = datos.groupby(['Season', 'Circuit'])['Laps'].max().rename('Max_Laps').reset_index()
datos = datos.merge(vueltas_max, on=['Season', 'Circuit'], how='left')
mask_finished = datos['Laps'] >= (datos['Max_Laps'] - 1)
datos = datos[mask_finished].copy()
datos.drop(columns=['Max_Laps'], inplace=True)
datos.drop(columns=["Location", "Country", "Date", "Round", "Race Name", "Time_of_race", 
                   "Total Pit Stops", "Fast Lap Attempts"], inplace=True, errors='ignore')

In [ ]:
# Normalizar tipos
datos['Pit_Lap'] = pd.to_numeric(datos['Pit_Lap'].replace('', np.nan), errors='coerce')
datos['Tire Compound'] = datos['Tire Compound'].astype(str).replace('nan', 'Unknown')
datos['Pit_Time'] = datos['Pit_Time'].astype(str)
mask_final_stint = datos['Pit_Time'] == "Final Stint"
datos.loc[mask_final_stint, 'Pit_Lap'] = datos.loc[mask_final_stint, 'Pit_Lap'].fillna(-1)

In [ ]:
print(f"Dataset limpio: {datos.shape}")

In [ ]:
# 2. SPLIT TEMPORAL
num_cols = ['TotalPitStops', 'AvgPitStopTime', 'Air_Temp_C', 'Track_Temp_C', 'Humidity_%', 
           'Wind_Speed_KMH', 'Pit_Lap', 'Driver Aggression Score', 'Tire Usage Aggression']
cat_cols = ['Driver', 'Constructor', 'Tire Compound', 'Circuit']

In [ ]:
train_df = datos[datos['Season'].between(2018, 2023)].copy()
test_df = datos[datos['Season'] == 2024].copy()
X_train = train_df[num_cols + cat_cols].copy()
y_train = train_df['Position']
X_test = test_df[num_cols + cat_cols].copy()
y_test = test_df['Position']
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# 3. MULTICOLINEALIDAD
print("\n=== 3. MULTICOLINEALIDAD ===")
corr_matrix = train_df[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Matriz de Correlación")
plt.tight_layout()
plt.savefig("plots/correlation_matrix.png", dpi=300)
plt.close()

In [ ]:
high_corr_pairs = [(corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]) 
                  for i in range(len(corr_matrix.columns)) 
                  for j in range(i+1, len(corr_matrix.columns)) 
                  if abs(corr_matrix.iloc[i, j]) > 0.8]
for var1, var2, corr in high_corr_pairs:
    print(f"🔴 {var1} ↔ {var2}: {corr:.3f}")
print("✓ correlation_matrix.png")

In [ ]:
# 4. PREPROCESADOR OPTIMIZADO
print("\n=== 4. PREPROCESADOR ===")
scalers = [StandardScaler(), MinMaxScaler(), RobustScaler()]
imputers = [SimpleImputer(strategy="mean"), SimpleImputer(strategy="median")]
mejor_config_pre = {'scaler': None, 'imputer': None, 'score': np.inf}

In [ ]:
tscv_pre = TimeSeriesSplit(n_splits=3)
for scaler in scalers:
    for imputer in imputers:
        num_trans = Pipeline([("imputer", imputer), ("scaler", scaler)])
        cat_trans = Pipeline([("enc", OneHotEncoder(handle_unknown="ignore"))])
        pre = ColumnTransformer([("num", num_trans, num_cols), ("cat", cat_trans, cat_cols)])
        pipe = Pipeline([("pre", pre), ("model", KNeighborsRegressor(n_neighbors=5))])
        scores = cross_val_score(pipe, X_train, y_train, cv=tscv_pre, 
                               scoring="neg_mean_absolute_error", n_jobs=-1)
        mae = -scores.mean()
        if mae < mejor_config_pre['score']:
            mejor_config_pre = {'scaler': scaler, 'imputer': imputer, 'score': mae}

In [ ]:
mejor_scaler = mejor_config_pre['scaler']
mejor_imputer = mejor_config_pre['imputer']
numerical_final = Pipeline([("imputer", mejor_imputer), ("scaler", mejor_scaler)])
categorical_final = Pipeline([("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocesador = ColumnTransformer([("num", numerical_final, num_cols), 
                                  ("cat", categorical_final, cat_cols)], sparse_threshold=0)

In [ ]:
# 5. ENTRENAMIENTO COMPLETO (TSCV + CV)
print("\n=== 5. ENTRENAMIENTO MODELOS (TSCV + CV) ===")
def exact_acc(y_t, y_p): return np.mean(y_t == np.round(y_p))
def tol_acc(y_t, y_p): return np.mean(np.abs(y_t - np.round(y_p)) <= 1)
scorer_exact = make_scorer(exact_acc)
scorer_tol = make_scorer(tol_acc)

In [ ]:
resultados_modelos = {}
tscv = TimeSeriesSplit(n_splits=4)

In [ ]:
# Random Forest
print(">> Random Forest")
rf_pipe = Pipeline([("pre", preprocesador), ("model", RandomForestRegressor(random_state=SEED, n_jobs=-1))])
rf_params = {"model__n_estimators": randint(200, 600), "model__max_depth": randint(10, 25),
            "model__min_samples_leaf": randint(1, 8), "model__max_features": ["sqrt", "log2"]}
rf_search = RandomizedSearchCV(rf_pipe, rf_params, n_iter=25, cv=3, 
                              scoring="neg_mean_absolute_error", random_state=SEED, n_jobs=-1)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_
acc_rf_exact = cross_val_score(best_rf, X_train, y_train, cv=3, scoring=scorer_exact, n_jobs=-1).mean()
resultados_modelos['RF'] = {'pipe': best_rf, 'mae_cv': -rf_search.best_score_, 'acc_exact': acc_rf_exact}

In [ ]:
# XGBoost
print(">> XGBoost")
xgb_pipe = Pipeline([("pre", preprocesador), ("model", XGBRegressor(random_state=SEED, n_jobs=-1, tree_method="hist"))])
xgb_params = {"model__n_estimators": randint(200, 500), "model__learning_rate": uniform(0.01, 0.2),
             "model__max_depth": randint(3, 10), "model__subsample": uniform(0.7, 0.3)}
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=30, cv=3, 
                               scoring="neg_mean_absolute_error", random_state=SEED, n_jobs=-1)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
acc_xgb_exact = cross_val_score(best_xgb, X_train, y_train, cv=3, scoring=scorer_exact, n_jobs=-1).mean()
resultados_modelos['XGB'] = {'pipe': best_xgb, 'mae_cv': -xgb_search.best_score_, 'acc_exact': acc_xgb_exact}

In [ ]:
# HistGradientBoosting
print(">> HistGradientBoosting")
hgb_pipe = Pipeline([("pre", preprocesador), ("model", HistGradientBoostingRegressor(random_state=SEED))])
hgb_params = {"model__max_iter": randint(200, 400), "model__learning_rate": uniform(0.05, 0.15),
             "model__max_depth": randint(5, 12)}
hgb_search = RandomizedSearchCV(hgb_pipe, hgb_params, n_iter=30, cv=3, 
                               scoring="neg_mean_absolute_error", random_state=SEED, n_jobs=-1)
hgb_search.fit(X_train, y_train)
best_hgb = hgb_search.best_estimator_
acc_hgb_exact = cross_val_score(best_hgb, X_train, y_train, cv=3, scoring=scorer_exact, n_jobs=-1).mean()
resultados_modelos['HGB'] = {'pipe': best_hgb, 'mae_cv': -hgb_search.best_score_, 'acc_exact': acc_hgb_exact}

In [ ]:
# 6. TABLA COMPARATIVA
print("\n=== TABLA COMPARATIVA ===")
print(f"{'Modelo':<10} {'MAE CV':<10} {'Acc Exacta':<12}")
print("-" * 40)
for name, res in resultados_modelos.items():
    print(f"{name:<10} {res['mae_cv']:<10.3f} {res['acc_exact']:<12.2%}")

In [ ]:
mejor_modelo_nombre = min(resultados_modelos, key=lambda k: resultados_modelos[k]['mae_cv'])
mejor_pipe = resultados_modelos[mejor_modelo_nombre]['pipe']
print(f"\n🏆 GANADOR: {mejor_modelo_nombre}")

In [ ]:
# 7. FEATURE IMPORTANCE
print(f"\n=== FEATURE IMPORTANCE {mejor_modelo_nombre} ===")
pre_step = mejor_pipe.named_steps['pre']
model_step = mejor_pipe.named_steps['model']
try:
    feat_names = pre_step.get_feature_names_out()
except:
    feat_names = [f'Var{i}' for i in range(100)]

In [ ]:
X_train_trans = pre_step.transform(X_train[:2000])  # Submalla para velocidad
if mejor_modelo_nombre == 'HGB':
    r = permutation_importance(model_step, X_train_trans, y_train[:2000], n_repeats=3, random_state=SEED)
    importancias = pd.Series(r.importances_mean, index=feat_names[:len(r.importances_mean)])
else:
    importancias = pd.Series(model_step.feature_importances_, index=feat_names[:len(model_step.feature_importances_)])

In [ ]:
top10 = importancias.nlargest(10)
for i, (var, imp) in enumerate(top10.items(), 1):
    print(f"{i:2d}. {str(var)[:40]} {imp:.4f}")

In [ ]:
top10.plot(kind='barh', color='skyblue')
plt.title(f"Top 10 Variables - {mejor_modelo_nombre}")
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig(f"plots/feature_importance_{mejor_modelo_nombre.lower()}.png", dpi=300)
plt.close()

Reemplaza la sección TEST 2024 (líneas 170-180 aprox) con esto:

In [ ]:
# 8. TEST 2024 CON MARGEN VISUAL ±1
print("\n=== TEST 2024 CON MARGEN VISUAL ±1 ===")
y_pred = mejor_pipe.predict(X_test)
mae_test = mean_absolute_error(y_test, y_pred)
print(f"MAE Test: {mae_test:.4f}")
print(f"Acc Exacta: {exact_acc(y_test, y_pred):.1%}")
print(f"Acc ±1: {tol_acc(y_test, y_pred):.1%}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, color='royalblue', s=50, edgecolors='darkblue', linewidth=0.5)
plt.plot([1, 20], [1, 20], 'r--', lw=3, label='Perfecto (pred = real)', alpha=0.9)

In [ ]:
# 🎯 LÍNEAS DE MARGEN ±1 POSICIONES
plt.plot([1, 20], [2, 21], 'orange', lw=2, linestyle=':', alpha=0.8, label='Margen +1')
plt.plot([1, 20], [0, 19], 'orange', lw=2, linestyle=':', alpha=0.8, label='Margen -1')

In [ ]:
# ZONA DE TOLERANCIA ±1 (relleno sombreado)
plt.fill_between([1, 20], [0, 19], [2, 21], color='orange', alpha=0.15, label='Zona ±1 posición')

In [ ]:
plt.xlabel('Posición Real 2024', fontsize=12, fontweight='bold')
plt.ylabel('Posición Predicha', fontsize=12, fontweight='bold')
plt.title(f'Predicción vs Realidad 2024 - {mejor_modelo_nombre}\nMAE: {mae_test:.2f}', fontsize=14, fontweight='bold')
plt.xlim(0.5, 20.5)
plt.ylim(0.5, 20.5)
plt.xticks(range(1, 21))
plt.yticks(range(1, 21))
plt.grid(True, linestyle=':', alpha=0.3)
plt.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('plots/pred_vs_real_2024.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ pred_vs_real_2024.png (con margen ±1 visual)")